# CourtVision — TrackNet Retrain (v5, self-contained)

Run this top-to-bottom on a **GPU** runtime (Runtime → Change runtime type → GPU).
Everything is embedded — you only supply your labeled data.

**What it does:** trains the ball detector at higher resolution (720×1280) on your
*real* hand labels, breaking the pseudo-label loop that caused the far-court blind
spot. Saves a checkpoint that drops straight into the CourtVision repo.

**You need:** a folder with `frames/frame_XXXXXX.jpg` (from
`extract_frames_hires.py`) and a `labels.csv` (`frame_idx,cx_norm,cy_norm`, from
`label_ball.py`). Put it in Google Drive.


## 1 · GPU check + imports

In [ ]:
!nvidia-smi -L
import torch, cv2, numpy as np, os, csv, math, random
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| cv2", cv2.__version__)
assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime → Change runtime type → GPU"
DEVICE = "cuda"


## 2 · Point at your data (Google Drive)

Mount Drive and set `DATA_DIR` to the folder containing `frames/` and `labels.csv`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# EDIT THIS to your data folder:
DATA_DIR = '/content/drive/MyDrive/courtvision/hires'

assert os.path.isdir(os.path.join(DATA_DIR, 'frames')), f"no frames/ in {DATA_DIR}"
assert os.path.exists(os.path.join(DATA_DIR, 'labels.csv')), f"no labels.csv in {DATA_DIR}"
n_frames = len(os.listdir(os.path.join(DATA_DIR, 'frames')))
n_rows = sum(1 for _ in open(os.path.join(DATA_DIR, 'labels.csv'))) - 1
print(f"OK — {n_frames} frames, {n_rows} label rows in {DATA_DIR}")


## 3 · Config

In [ ]:
INPUT_H, INPUT_W = 720, 1280   # higher-res: the far ball spans real pixels now
SIGMA      = 8.0               # Gaussian target radius (px, at this resolution). Keep ~ball size — do NOT scale up.
EPOCHS     = 60
BATCH      = 4                 # lower if you hit out-of-memory
LR         = 1e-4
VAL_FRAC   = 0.15
BLOCK      = 30                # leak-free split: whole blocks of frames held out
POS_WEIGHT = 30.0             # ball pixels are <0.3% of the image; weight them up
SAVE_PATH  = '/content/tracknet_hires.pt'


## 4 · Model (TrackNet — 3-frame U-Net heatmap)
Exact architecture used by the repo, so the checkpoint loads directly.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class _CBR(nn.Module):
    def __init__(self, ci, co, k=3, p=1):
        super().__init__()
        self.seq = nn.Sequential(nn.Conv2d(ci, co, k, padding=p, bias=False),
                                 nn.BatchNorm2d(co), nn.ReLU(inplace=True))
    def forward(self, x): return self.seq(x)

def _enc(ci, co, n):
    layers = [_CBR(ci, co)] + [_CBR(co, co) for _ in range(n - 1)]
    return nn.Sequential(*layers)

class TrackNet(nn.Module):
    """3 RGB frames (9ch) in -> 1 heatmap out."""
    INPUT_H, INPUT_W = 720, 1280
    def __init__(self, dropout=0.0):
        super().__init__()
        self.enc1 = _enc(9, 64, 2);  self.enc2 = _enc(64, 128, 2)
        self.enc3 = _enc(128, 256, 3); self.enc4 = _enc(256, 512, 3)
        self.pool = nn.MaxPool2d(2, 2); self.drop = nn.Dropout2d(dropout)
        self.dec4 = _enc(512 + 256, 256, 2)
        self.dec3 = _enc(256 + 128, 128, 2)
        self.dec2 = _enc(128 + 64, 64, 2)
        self.out_conv = nn.Conv2d(64, 1, 1)
    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2)); e4 = self.drop(self.enc4(self.pool(e3)))
        d4 = F.interpolate(e4, size=e3.shape[2:], mode='bilinear', align_corners=False)
        d4 = self.drop(self.dec4(torch.cat([d4, e3], 1)))
        d3 = F.interpolate(d4, size=e2.shape[2:], mode='bilinear', align_corners=False)
        d3 = self.drop(self.dec3(torch.cat([d3, e2], 1)))
        d2 = F.interpolate(d3, size=e1.shape[2:], mode='bilinear', align_corners=False)
        d2 = self.dec2(torch.cat([d2, e1], 1))
        return self.out_conv(d2)

def gaussian_heatmap(cx, cy, h, w, sigma):
    if cx < 0 or cy < 0:
        return np.zeros((h, w), np.float32)
    xs = np.arange(w, dtype=np.float32); ys = np.arange(h, dtype=np.float32)
    X, Y = np.meshgrid(xs, ys)
    return np.exp(-(((X - cx * w) ** 2 + (Y - cy * h) ** 2) / (2 * sigma ** 2))).astype(np.float32)


## 5 · Dataset + augmentation
Augmentation (brightness, motion blur, h-flip) is applied on the training split only, to help the model generalise.

In [ ]:
class BallDataset(torch.utils.data.Dataset):
    def __init__(self, data_dir, augment=False):
        self.dir = os.path.join(data_dir, 'frames'); self.aug = augment
        self.rows = []
        edge = 0.02
        with open(os.path.join(data_dir, 'labels.csv')) as f:
            for r in csv.DictReader(f):
                cx, cy = float(r['cx_norm']), float(r['cy_norm'])
                if cx >= 0 and (cx < edge or cx > 1 - edge or cy < edge or cy > 1 - edge):
                    cx = cy = -1.0
                self.rows.append((int(r['frame_idx']), cx, cy))
    def __len__(self): return len(self.rows)
    def _load(self, idx):
        p = os.path.join(self.dir, f'frame_{idx:06d}.jpg')
        im = cv2.imread(p)
        if im is None: return np.zeros((INPUT_H, INPUT_W, 3), np.float32)
        im = cv2.cvtColor(cv2.resize(im, (INPUT_W, INPUT_H)), cv2.COLOR_BGR2RGB)
        return im.astype(np.float32) / 255.0
    def __getitem__(self, i):
        t, cx, cy = self.rows[i]
        f = [self._load(max(0, t - 2)), self._load(max(0, t - 1)), self._load(t)]
        if self.aug:
            a = random.uniform(0.8, 1.2); b = random.uniform(-0.08, 0.08)   # brightness/contrast
            f = [np.clip(x * a + b, 0, 1) for x in f]
            if random.random() < 0.3:                                        # motion blur
                k = random.choice([3, 5]); ker = np.zeros((k, k), np.float32); ker[k // 2] = 1.0 / k
                f = [cv2.filter2D(x, -1, ker) for x in f]
            if random.random() < 0.5:                                        # horizontal flip
                f = [x[:, ::-1].copy() for x in f]
                if cx >= 0: cx = 1.0 - cx
        x = torch.from_numpy(np.concatenate(f, axis=2)).permute(2, 0, 1).float()
        y = torch.from_numpy(gaussian_heatmap(cx, cy, INPUT_H, INPUT_W, SIGMA)).unsqueeze(0)
        return x, y

def split_by_blocks(ds, val_frac=VAL_FRAC, block=BLOCK, seed=42):
    n = len(ds); nb = max(2, (n + block - 1) // block); ids = list(range(nb))
    random.Random(seed).shuffle(ids); vb = set(ids[:max(1, round(val_frac * nb))])
    tr = [i for i in range(n) if (i // block) not in vb]
    va = [i for i in range(n) if (i // block) in vb]
    return torch.utils.data.Subset(ds, tr), torch.utils.data.Subset(ds, va)


## 6 · Train (with a real detection-rate metric on val)

In [ ]:
from torch.utils.data import DataLoader

full_train = BallDataset(DATA_DIR, augment=True)
full_val   = BallDataset(DATA_DIR, augment=False)
tr_idx, va_idx = split_by_blocks(full_train)
train_ds = torch.utils.data.Subset(full_train, tr_idx.indices)
val_ds   = torch.utils.data.Subset(full_val,   va_idx.indices)
print(f"train {len(train_ds)}  |  val {len(val_ds)}  |  {INPUT_W}x{INPUT_H}  sigma {SIGMA}")

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

model = TrackNet().to(DEVICE)
crit  = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))
opt   = torch.optim.Adam(model.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)

def val_detect_rate(tol_px=8):
    """Fraction of val balls whose predicted heatmap peak is within tol_px."""
    model.eval(); hit = tot = 0
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE); pr = torch.sigmoid(model(x)).cpu().numpy()
            gt = y.numpy()
            for b in range(pr.shape[0]):
                if gt[b, 0].max() <= 0: continue    # no-ball frame
                tot += 1
                gy, gx = np.unravel_index(gt[b, 0].argmax(), gt[b, 0].shape)
                py, px = np.unravel_index(pr[b, 0].argmax(), pr[b, 0].shape)
                if math.hypot(px - gx, py - gy) <= tol_px and pr[b, 0].max() >= 0.5: hit += 1
    return hit / max(1, tot)

best = 1e9
for ep in range(1, EPOCHS + 1):
    model.train(); tl = 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        loss = crit(model(x), y); opt.zero_grad(); loss.backward(); opt.step()
        tl += loss.item() * x.size(0)
    tl /= len(train_ds)
    model.eval(); vl = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE); vl += crit(model(x), y).item() * x.size(0)
    vl /= len(val_ds); sched.step(vl)
    dr = val_detect_rate()
    print(f"epoch {ep:3d}/{EPOCHS}  train {tl:.5f}  val {vl:.5f}  val-detect@8px {dr*100:5.1f}%")
    if vl < best:
        best = vl
        torch.save({'model': model.state_dict(), 'input_h': INPUT_H,
                    'input_w': INPUT_W, 'sigma': SIGMA}, SAVE_PATH)
        print(f"   ✓ saved best → {SAVE_PATH}")
print("done. best val loss", round(best, 5))


## 7 · Download the model

In [ ]:
# save a copy to Drive and offer a direct download
import shutil
drive_copy = os.path.join(DATA_DIR, 'tracknet_hires.pt')
shutil.copy(SAVE_PATH, drive_copy); print("copied to", drive_copy)
from google.colab import files
files.download(SAVE_PATH)


## 8 · Next — the eval gate (back in the repo)

Put `tracknet_hires.pt` in `models/`, then **promote it only if it beats the
incumbent** on held-out labels:

```bash
python eval/diagnose_ball.py --tracknet models/tracknet_hires.pt   # vs tracknet_v4_best.pt
python eval/line_call_eval.py score eval/preds.json eval/labels.json
```

Keep the new model only if far-half recall and line-call accuracy improve without
adding false positives. Then run `python main.py --video your_clip.mp4`.
